# Huấn luyện U-MobileViT cho Phân vùng Ngữ nghĩa Đa lớp trên COCO 2017 (10 lớp chính)

Notebook này triển khai quá trình huấn luyện mô hình **U-MobileViT** cho tác vụ
**phân vùng ngữ nghĩa (semantic segmentation)** trên **10 lớp đối tượng chính**
của bộ dữ liệu COCO 2017. Kiến trúc mạng được nhập trực tiếp từ mô-đun
`models/u_mobilevit_net/`, đảm bảo tính nhất quán giữa định nghĩa và huấn luyện.

Mười lớp được chọn bao gồm các đối tượng phổ biến và đa dạng về mặt thị giác:
**person, bicycle, car, motorcycle, dog, cat, chair, dining table, cup, bottle**
— phủ các nhóm người, phương tiện, động vật, đồ nội thất và đồ dùng nhà bếp.

Các đặc điểm chính của notebook:

- **Đầu ra 10 kênh**: mỗi kênh tương ứng với một lớp đối tượng, đi qua hàm
  softmax để thu được phân phối xác suất trên 10 lớp.
- **Hàm mất mát kết hợp**: Cross-Entropy có `ignore_index=255` cho pixel nền
  kết hợp với Dice Loss đa lớp, cân bằng giữa tối ưu từng pixel và tối ưu
  vùng chồng lấp.
- **Chỉ số đánh giá mIoU**: mean Intersection over Union được tính trên 10 lớp,
  cung cấp bức tranh toàn diện về chất lượng phân vùng.
- **Mặt nạ đa lớp**: mỗi pixel được gán chỉ số lớp (0–9), pixel nền được gán
  nhãn 255 và bị bỏ qua trong quá trình tính loss cũng như đánh giá.

**Yêu cầu**: thực thi từ thư mục gốc của dự án với biến môi trường `PYTHONPATH`
trỏ đến thư mục gốc nhằm đảm bảo khả năng import các mô-đun `models` và `cv_nets`.


## 1. Thư viện và Phụ thuộc

Các thư viện cần thiết cho xử lý dữ liệu (COCO API, PIL), huấn luyện (PyTorch,
torchvision), và trực quan hóa (Matplotlib).


In [ ]:
import os
import json
import random
import argparse

import numpy as np
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from PIL import Image
from tqdm import tqdm
from pycocotools.coco import COCO
import matplotlib.pyplot as plt

from models.u_mobilevit_net.u_models import umobilevit, UMobileViT


## 2. Cấu hình Thực nghiệm

Các siêu tham số cho quá trình huấn luyện bao gồm: kích thước ảnh đầu vào, kích
thước batch, số worker, tổng số epoch, và các tham số kiểm soát early-stopping.
`NUM_CLASSES = 10` tương ứng với 10 lớp đối tượng chính được chọn từ COCO 2017;
`IGNORE_INDEX = 255` dùng để đánh dấu pixel nền không thuộc lớp nào.


In [ ]:
IMAGE_SIZE = (320, 320)
BATCH_SIZE = 48
NUM_WORKERS = 16
EPOCH = 100

NUM_CLASSES = 10         # 10 lớp đối tượng chính của COCO
HEAD_TYPE = "single"     # ContextAwareSegHead đơn nhánh
IGNORE_INDEX = 255       # nhãn bỏ qua cho pixel nền

# 10 lớp: person, bicycle, car, motorcycle, dog, cat, chair, dining table, cup, bottle
COCO_CAT_IDS = [1, 2, 3, 4, 18, 17, 62, 67, 47, 44]
CAT_ID_TO_INDEX = {cat_id: idx for idx, cat_id in enumerate(COCO_CAT_IDS)}

TRAIN_IMG_DIR = './data/COCO/train2017'
TRAIN_ANN_FILE = './data/COCO/annotations/instances_train2017.json'
VAL_IMG_DIR = './data/COCO/val2017'
VAL_ANN_FILE = './data/COCO/annotations/instances_val2017.json'

SAVE_MODEL_DIR = './models/u_mobilevit_net'
os.makedirs(SAVE_MODEL_DIR, exist_ok=True)

patience = 15
min_epochs_before_stop = 30

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {device})")


## 3. Khởi tạo Mô hình và Kiểm tra Tính nhất quán

Mô hình `UMobileViT` được khởi tạo thông qua hàm factory `umobilevit()`, sử dụng
`ContextAwareSegHead` với `out_channels=10`. Mỗi kênh đầu ra đại diện cho logit
của một lớp đối tượng. Trước khi nạp dữ liệu, một tensor giả lập được truyền qua
mạng nhằm xác nhận rằng kiến trúc được dựng chính xác và đầu ra có kích thước
$(B, 10, H, W)$ như kỳ vọng.


In [ ]:
# Sinh tham số mặc định cho kiến trúc
_parser = argparse.ArgumentParser()
UMobileViT.add_arguments(_parser)
opts, _ = _parser.parse_known_args([])

def build_model():
    return umobilevit(opts=opts, head=HEAD_TYPE, out_channels=NUM_CLASSES)

model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"U-MobileViT | head={HEAD_TYPE} | out_channels={NUM_CLASSES} | tham số={n_params/1e6:.3f} M")

# Smoke test: lan truyền xuôi với tensor giả lập
model.eval()
with torch.no_grad():
    _dummy = torch.randn(2, 3, *IMAGE_SIZE)
    _out = model(_dummy)
assert _out.shape == (2, NUM_CLASSES, *IMAGE_SIZE), f"Sai kích thước đầu ra: {_out.shape}"
print(f"[smoke test] Đạt — input {tuple(_dummy.shape)} → output {tuple(_out.shape)}")


## 4. Bộ dữ liệu Phân vùng Ngữ nghĩa COCO 2017 (10 lớp)

Lớp `COCOSegmentationDataset` kế thừa `torch.utils.data.Dataset`, thực hiện các
công việc sau:

1. **Nạp ảnh và annotation**: Đọc ảnh từ `train2017/` hoặc `val2017/` và tệp
   annotation `instances_*.json` tương ứng.
2. **Xây dựng mặt nạ đa lớp**: Mỗi pixel được gán chỉ số lớp (0–9) dựa trên
   ánh xạ từ `category_id` của COCO sang chỉ số liên tục; pixel không thuộc bất
   kỳ đối tượng nào trong 10 lớp được chọn sẽ được gán nhãn `IGNORE_INDEX` (255).
3. **Tăng cường dữ liệu đồng bộ**: Các phép biến đổi hình học (resize, xoay,
   cắt, lật ngang) được áp dụng đồng thời lên cả ảnh và mặt nạ, trong đó mặt nạ
   luôn sử dụng phép nội suy `NEAREST` để bảo toàn giá trị nguyên của nhãn lớp.
   Các phép biến đổi màu sắc (ColorJitter, GaussianBlur) chỉ được áp dụng trên ảnh.

Danh sách 10 lớp được chọn: **person, bicycle, car, motorcycle, dog, cat, chair,
dining table, cup, bottle**. Ánh xạ `CAT_ID_TO_INDEX` chuyển đổi `category_id`
của COCO sang chỉ số 0–9 cho các lớp này; các annotation thuộc lớp khác bị bỏ qua.


In [ ]:
class COCOSegmentationDataset(Dataset):
    """Dataset COCO cho phân vùng ngữ nghĩa 10 lớp, có tăng cường đồng bộ ảnh–mask."""
    def __init__(self, img_dir, ann_file, image_size=IMAGE_SIZE, is_train=True):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.image_size = image_size
        self.is_train = is_train
        self.img_ids = self.coco.getImgIds()

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        image = Image.open(img_path).convert('RGB')

        # Xây dựng mặt nạ đa lớp: nền = IGNORE_INDEX, đối tượng = chỉ số 0–9
        mask = np.full((img_info['height'], img_info['width']), IGNORE_INDEX, dtype=np.uint8)
        ann_ids = self.coco.getAnnIds(imgIds=img_id, iscrowd=False)
        anns = self.coco.loadAnns(ann_ids)
        for ann in anns:
            cat_id = ann['category_id']
            if cat_id in CAT_ID_TO_INDEX:
                class_idx = CAT_ID_TO_INDEX[cat_id]
                ann_mask = self.coco.annToMask(ann)
                mask[ann_mask > 0] = class_idx

        mask = Image.fromarray(mask, mode='L')  # uint8, giá trị 0–9 và 255

        # Tăng cường dữ liệu đồng bộ ảnh–mask
        if self.is_train:
            short_edge = random.randint(min(self.image_size), int(min(self.image_size) * 1.5))
            image = TF.resize(image, short_edge, interpolation=InterpolationMode.BILINEAR)
            mask = TF.resize(mask, short_edge, interpolation=InterpolationMode.NEAREST)

            if random.random() > 0.5:
                angle = random.uniform(-15, 15)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=IGNORE_INDEX)

            i, j, h, w = T.RandomCrop.get_params(image, output_size=self.image_size)
            image = TF.crop(image, i, j, h, w)
            mask = TF.crop(mask, i, j, h, w)

            if random.random() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)
            if random.random() > 0.5:
                image = T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)(image)
            if random.random() > 0.7:
                image = T.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 2.0))(image)
        else:
            short_edge = min(self.image_size)
            image = TF.resize(image, short_edge, interpolation=InterpolationMode.BILINEAR)
            mask = TF.resize(mask, short_edge, interpolation=InterpolationMode.NEAREST)
            image = TF.center_crop(image, self.image_size)
            mask = TF.center_crop(mask, self.image_size)

        image_tensor = TF.to_tensor(image)
        image_tensor = TF.normalize(image_tensor, mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
        # Chuyển mặt nạ thành LongTensor (giữ nguyên giá trị lớp)
        mask_tensor = torch.from_numpy(np.array(mask)).long()
        return image_tensor, mask_tensor

## 5. Chỉ số Đánh giá: Mean Intersection over Union (mIoU)

Hàm `calculate_miou` tính trung bình IoU trên tất cả các lớp có xuất hiện trong
batch. Với mỗi lớp $c$, IoU được định nghĩa là:

$$\text{IoU}_c = \frac{|P_c \cap G_c|}{|P_c \cup G_c|}$$

trong đó $P_c$ là tập pixel được dự đoán thuộc lớp $c$ và $G_c$ là tập pixel
thực sự thuộc lớp $c$ theo ground-truth. Pixel mang nhãn `IGNORE_INDEX` bị loại
khỏi quá trình tính toán. Giá trị mIoU cuối cùng là trung bình cộng IoU trên
các lớp có xuất hiện ít nhất một pixel trong batch.


In [ ]:
@torch.no_grad()
def calculate_miou(preds, targets, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
    """Tính mean IoU cho phân vùng ngữ nghĩa đa lớp.

    Chỉ tính IoU cho các lớp **có xuất hiện** trong batch (target_c.sum() > 0).
    Các lớp vắng mặt bị bỏ qua để tránh IoU=0 gây nhiễu kết quả trung bình.

    Args:
        preds:       (B, C, H, W) logits.
        targets:     (B, H, W) chỉ số lớp (0..C-1, ignore_index cho nền).
        num_classes: số lớp đối tượng.
        ignore_index: nhãn pixel bị bỏ qua.
    Returns:
        float: mIoU trung bình trên các lớp có xuất hiện trong batch.
    """
    preds = preds.argmax(dim=1)                       # (B, H, W)
    valid_mask = (targets != ignore_index)            # loại bỏ pixel nền khỏi đánh giá
    ious = []
    for c in range(num_classes):
        target_c = (targets == c)                     # (B, H, W), bool — True chỉ ở pixel thuộc lớp c
        if target_c.sum() == 0:
            # Lớp không xuất hiện trong batch → bỏ qua
            continue
        pred_c = (preds == c) & valid_mask            # (B, H, W), bool — dự đoán lớp c trên foreground
        intersection = (pred_c & target_c).float().sum()
        union = (pred_c | target_c).float().sum()
        ious.append(((intersection + 1e-6) / (union + 1e-6)).item())
    return sum(ious) / len(ious) if ious else 0.0

## 6. Trực quan hóa Dữ liệu Huấn luyện

Hiển thị một số mẫu từ DataLoader bao gồm ảnh gốc và mặt nạ đa lớp được tô màu
theo bảng màu riêng cho 10 lớp. Bảng màu được chọn thủ công để các lớp dễ phân
biệt trực quan. Pixel nền (không thuộc lớp nào) được hiển thị bằng màu đen.


In [ ]:
# Bảng màu thủ công cho 10 lớp COCO (dễ phân biệt trực quan)
COCO_PALETTE = np.array([
    [220,  20,  60],  # 0: person     - đỏ thẫm
    [255, 165,   0],  # 1: bicycle    - cam
    [  0, 100, 255],  # 2: car        - xanh dương
    [255, 255,   0],  # 3: motorcycle - vàng
    [148,   0, 211],  # 4: dog        - tím
    [ 50, 205,  50],  # 5: cat        - xanh lá
    [139,  69,  19],  # 6: chair      - nâu
    [  0, 206, 209],  # 7: dining table - xanh ngọc
    [255,  20, 147],  # 8: cup        - hồng
    [128, 128, 128],  # 9: bottle     - xám
], dtype=np.uint8)

CATEGORY_NAMES = ['person', 'bicycle', 'car', 'motorcycle', 'dog', 'cat', 'chair', 'dining table', 'cup', 'bottle']


def label_to_color(mask, palette=COCO_PALETTE, ignore_index=IGNORE_INDEX):
    """Chuyển mặt nạ chỉ số lớp (H, W) sang ảnh màu RGB (H, W, 3)."""
    h, w = mask.shape
    color = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_idx in range(palette.shape[0]):
        color[mask == cls_idx] = palette[cls_idx]
    color[mask == ignore_index] = [0, 0, 0]  # nền: đen
    return color


def visualize_datasets(dataloader, num_samples=8):
    """Hiển thị num_samples mẫu (ảnh | mặt nạ) từ dataloader."""
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225])
    fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(20, 14))
    plt.subplots_adjust(wspace=0.1, hspace=0.3)
    shown = 0
    for images, masks in dataloader:
        for i in range(images.size(0)):
            if shown >= num_samples:
                break
            row, col = shown // 2, (shown % 2) * 2
            img = np.clip(inv_normalize(images[i]).permute(1, 2, 0).numpy(), 0, 1)
            mask_color = label_to_color(masks[i].numpy())
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"Mẫu {shown+1}: Ảnh")
            axes[row, col].axis('off')
            axes[row, col+1].imshow(mask_color)
            axes[row, col+1].set_title("Mặt nạ (GT)")
            axes[row, col+1].axis('off')
            shown += 1
        if shown >= num_samples:
            break
    plt.suptitle("Dữ liệu Mẫu — Ảnh gốc & Mặt nạ Đa lớp (10 lớp)", fontsize=16, y=0.92)
    plt.show()


## 7. Nạp Dữ liệu Huấn luyện và Đánh giá

Khởi tạo các đối tượng `DataLoader` cho tập huấn luyện và tập đánh giá. Dữ liệu
huấn luyện được xáo trộn (`shuffle=True`) trong khi dữ liệu đánh giá giữ nguyên
thứ tự để kết quả có thể tái lập.


In [ ]:
print("Đang tải COCO Train2017 Annotations...")
train_dataset = COCOSegmentationDataset(TRAIN_IMG_DIR, TRAIN_ANN_FILE, image_size=IMAGE_SIZE, is_train=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
visualize_datasets(train_loader)


In [ ]:
val_dataset = COCOSegmentationDataset(VAL_IMG_DIR, VAL_ANN_FILE, image_size=IMAGE_SIZE, is_train=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
visualize_datasets(val_loader)  # hiển thị mẫu đánh giá để tham khảo

In [ ]:
print(f"Tập Huấn luyện: {len(train_dataset):,} ảnh")
print(f"Tập Đánh giá:   {len(val_dataset):,} ảnh")


## 8. Hàm Mất mát Kết hợp: Cross-Entropy và Dice Loss Đa lớp

Hàm mất mát được thiết kế dưới dạng tổ hợp có trọng số của hai thành phần:

1. **Cross-Entropy Loss** ($\mathcal{L}_{\text{CE}}$): hàm mất mát tiêu chuẩn
   cho bài toán phân loại đa lớp, hoạt động trên từng pixel độc lập. Sử dụng
   `ignore_index=255` để bỏ qua pixel nền, giúp mô hình tập trung học các vùng
   có đối tượng.

2. **Dice Loss Đa lớp** ($\mathcal{L}_{\text{Dice}}$): tối ưu trực tiếp hệ số
   Dice — một chỉ số đo độ chồng lấp giữa vùng dự đoán và vùng ground-truth —
   trên từng lớp. Kết quả cuối cùng là trung bình cộng Dice Loss trên các lớp
   có xuất hiện trong batch, giúp giảm thiểu hiện tượng mất cân bằng giữa các
   lớp có tần suất xuất hiện khác nhau.

Công thức tổng quát:
$$\mathcal{L} = (1 - \lambda) \cdot \mathcal{L}_{\text{CE}} + \lambda \cdot \mathcal{L}_{\text{Dice}}$$

với $\lambda$ là trọng số cân bằng giữa hai thành phần (mặc định $\lambda = 0.5$).


In [ ]:
class MultiClassSegLoss(nn.Module):
    """Hàm mất mát kết hợp Cross-Entropy và Dice cho phân vùng ngữ nghĩa đa lớp.

    Args:
        ce_weight:    trọng số cho Cross-Entropy.
        dice_weight:  trọng số cho Dice Loss.
        num_classes:  số lớp đối tượng.
        ignore_index: nhãn pixel bị bỏ qua khi tính loss.
    """
    def __init__(self, ce_weight=0.5, dice_weight=0.5, num_classes=NUM_CLASSES,
                 ignore_index=IGNORE_INDEX):
        super().__init__()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.num_classes = num_classes
        self.ignore_index = ignore_index

    def forward(self, inputs, targets, smooth=1e-6):
        # inputs:  (B, C, H, W) logits (có thể float16 từ autocast)
        # targets: (B, H, W) chỉ số lớp (LongTensor)
        inputs = inputs.float()
        targets = targets.long()

        # ── Cross-Entropy (bỏ qua pixel nền) ──
        ce = F.cross_entropy(inputs, targets, ignore_index=self.ignore_index)

        # ── Dice Loss đa lớp ──
        probs = F.softmax(inputs, dim=1)                     # (B, C, H, W)
        valid_mask = (targets != self.ignore_index).float()   # (B, H, W), 1.0 ở foreground

        dice_sum = torch.tensor(0.0, device=inputs.device, dtype=inputs.dtype)
        present = 0
        for c in range(self.num_classes):
            target_c = (targets == c).float()                 # (B, H, W), 1.0 ở pixel lớp c
            n_target = target_c.sum()
            if n_target == 0:
                # Bỏ qua lớp không xuất hiện trong batch
                continue

            pred_c = probs[:, c, :, :] * valid_mask           # zero-out background
            intersection = (pred_c * target_c).sum()
            union = pred_c.sum() + target_c.sum()

            # Dice cho lớp c (đã smooth, luôn trong [0, 1])
            dice_c = (2.0 * intersection + smooth) / (union + smooth)
            dice_sum += 1.0 - dice_c
            present += 1

        dice = dice_sum / max(present, 1)
        combined = self.ce_weight * ce + self.dice_weight * dice
        return combined, ce.detach(), dice.detach()

## 9. Thiết lập Huấn luyện

Cấu hình bộ tối ưu **Adam** với tốc độ học khởi tạo $5 \times 10^{-4}$ (đã giảm
từ $10^{-3}$ để ổn định FP16), bộ lập lịch **ReduceLROnPlateau** giảm tốc độ học
khi `val_loss` bão hòa, và huấn luyện với độ chính xác hỗn hợp (**Automatic Mixed
Precision — AMP**) khi có GPU nhằm tăng tốc độ và giảm bộ nhớ. Trước khi huấn
luyện, tất cả trọng số mô hình được kiểm tra NaN/Inf để phát hiện sớm lỗi khởi tạo.

In [ ]:
model = model.to(device)

# [FIX NaN] Bật cudnn benchmark để tối ưu hiệu năng trên GPU
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

# [FIX NaN] Kiểm tra trọng số mô hình trước khi huấn luyện — phát hiện sớm
# lỗi khởi tạo hoặc NaN từ checkpoint cũ.
nan_params = []
for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        nan_params.append(name)
if nan_params:
    raise RuntimeError(f"[LỖI NGHIÊM TRỌNG] {len(nan_params)} tham số chứa NaN/Inf "
                       f"ngay từ khi khởi tạo: {nan_params[:5]}...")
print("[OK] Tất cả trọng số mô hình hợp lệ (không NaN/Inf).")

scaler = torch.amp.GradScaler(enabled=device.type == 'cuda')
criterion = MultiClassSegLoss(ce_weight=0.5, dice_weight=0.5)
# [FIX NaN] Giảm learning rate từ 1e-3 → 5e-4 để tránh bước cập nhật quá lớn
# gây overflow trong separable attention khi chạy FP16.
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                                  patience=4, min_lr=1e-6)

GRAD_CLIP_NORM = 1.0                # chống gradient explosion
# [FIX NaN] Đã sửa root cause (softmax FP16 overflow trong separable_attention_forward).
# Để DETECT_ANOMALY=False để tăng tốc. Bật lại True nếu vẫn gặp NaN.
DETECT_ANOMALY = False
torch.autograd.set_detect_anomaly(DETECT_ANOMALY)

history = {
    'train_loss': [], 'train_ce': [], 'train_dice': [],
    'val_loss': [], 'val_ce': [], 'val_dice': [], 'val_miou': [],
}
best_val_loss = float('inf')
early_stop_counter = 0


def _has_nan_or_inf(*tensors):
    """Kiểm tra xem có tensor nào chứa NaN hoặc Inf không."""
    for t in tensors:
        if t is None:
            continue
        if not torch.isfinite(t).all():
            return True
    return False


def train_one_epoch(epoch):
    """Huấn luyện một epoch, trả về (loss, ce, dice) trung bình."""
    model.train()
    running_loss, running_ce, running_dice = 0.0, 0.0, 0.0
    skipped = 0
    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCH} [Train]", unit="batch")
    for images, masks in bar:
        images, masks = images.to(device), masks.to(device)

        # Kiểm tra input
        if _has_nan_or_inf(images, masks):
            print("[CẢNH BÁO] NaN/Inf trong input batch — bỏ qua")
            skipped += 1
            continue

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=device.type == 'cuda'):
            outputs = model(images)

        # Phát hiện NaN từ model
        if _has_nan_or_inf(outputs):
            print("[CẢNH BÁO] NaN/Inf trong model outputs — bỏ qua batch")
            skipped += 1
            continue

        loss, ce, dice = criterion(outputs, masks)

        # Phát hiện NaN từ loss
        if _has_nan_or_inf(loss):
            print("[CẢNH BÁO] NaN/Inf trong loss — bỏ qua batch")
            skipped += 1
            continue

        scaler.scale(loss).backward()

        # Gradient clipping trước optimizer.step()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_ce += ce.item()
        running_dice += dice.item()
        bar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{dice.item():.4f}")

    n = len(train_loader) - skipped
    if skipped:
        print(f"[Train] Đã bỏ qua {skipped}/{len(train_loader)} batch do NaN/Inf")
    return running_loss / max(n, 1), running_ce / max(n, 1), running_dice / max(n, 1)


@torch.no_grad()
def evaluate(epoch):
    """Đánh giá một epoch, trả về (loss, ce, dice, miou) trung bình."""
    model.eval()
    loss_sum, ce_sum, dice_sum, miou_sum = 0.0, 0.0, 0.0, 0.0
    skipped = 0
    bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCH} [Val]  ", unit="batch",
               leave=False)
    for images, masks in bar:
        images, masks = images.to(device), masks.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=device.type == 'cuda'):
            outputs = model(images)

        if _has_nan_or_inf(outputs):
            skipped += 1
            continue

        loss, ce, dice = criterion(outputs, masks)
        if _has_nan_or_inf(loss):
            skipped += 1
            continue

        loss_sum += loss.item()
        ce_sum += ce.item()
        dice_sum += dice.item()
        miou_sum += calculate_miou(outputs, masks)

    n = len(val_loader) - skipped
    if skipped:
        print(f"[Val] Đã bỏ qua {skipped}/{len(val_loader)} batch do NaN/Inf")
    return loss_sum / max(n, 1), ce_sum / max(n, 1), dice_sum / max(n, 1), miou_sum / max(n, 1)

## 10. Vòng lặp Huấn luyện

Huấn luyện với cơ chế **early stopping**: nếu `val_loss` không cải thiện sau
`patience` epoch liên tiếp (tính từ epoch thứ `min_epochs_before_stop`), quá
trình huấn luyện sẽ tự động dừng nhằm tránh over-fitting và tiết kiệm tài nguyên
tính toán. Tại mỗi epoch, mô hình có `val_loss` tốt nhất được lưu vào checkpoint
riêng, trong khi checkpoint cuối cùng luôn được duy trì để có thể phục hồi trạng
thái huấn luyện.


In [ ]:
best_path = os.path.join(SAVE_MODEL_DIR, "best_u_mobilevit_net_10cls.pth")
last_path = os.path.join(SAVE_MODEL_DIR, "last_u_mobilevit_net_10cls.pth")

for epoch in range(EPOCH):
    avg_train_loss, avg_train_ce, avg_train_dice = train_one_epoch(epoch)
    avg_val_loss, avg_val_ce, avg_val_dice, avg_val_miou = evaluate(epoch)

    history['train_loss'].append(avg_train_loss)
    history['train_ce'].append(avg_train_ce)
    history['train_dice'].append(avg_train_dice)
    history['val_loss'].append(avg_val_loss)
    history['val_ce'].append(avg_val_ce)
    history['val_dice'].append(avg_val_dice)
    history['val_miou'].append(avg_val_miou)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"→ Epoch {epoch+1:03d} | LR: {current_lr:.6f} | "
          f"Train: loss={avg_train_loss:.4f} ce={avg_train_ce:.4f} "
          f"dice={avg_train_dice:.4f} | "
          f"Val: loss={avg_val_loss:.4f} ce={avg_val_ce:.4f} "
          f"dice={avg_val_dice:.4f} mIoU={avg_val_miou:.4f}")
    scheduler.step(avg_val_loss)

    ckpt = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': avg_val_loss, 'val_miou': avg_val_miou, 'history': history,
    }

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stop_counter = 0
        torch.save(ckpt, best_path)
        print(f"✓ Đã cập nhật mô hình tối ưu tại: {best_path}\n")
    else:
        print()
        if (epoch + 1) > min_epochs_before_stop:
            early_stop_counter += 1
            print(f"[Cảnh báo] Val Loss không giảm. "
                  f"{early_stop_counter}/{patience} (Early Stopping)")
            if early_stop_counter >= patience:
                print(f"==== EARLY STOPPING tại Epoch {epoch+1} ====")
                torch.save(ckpt, last_path)
                break

    torch.save(ckpt, last_path)


## 11. Lưu và Trực quan hóa Lịch sử Huấn luyện

Toàn bộ diễn biến của loss (Cross-Entropy, Dice, Combined) và mIoU trên tập
huấn luyện và đánh giá được lưu ra tệp JSON và hiển thị dưới dạng biểu đồ
4 góc phần tư, cung cấp cái nhìn toàn diện về quá trình hội tụ của mô hình.


In [ ]:
train_his_file = os.path.join(SAVE_MODEL_DIR, "training_history_10cls.json")
with open(train_his_file, "w") as f:
    json.dump(history, f, indent=4)
print(f"Đã xuất lịch sử huấn luyện ra {train_his_file}.")


def plot_training_history(history_source):
    """Vẽ biểu đồ diễn biến loss và mIoU trong quá trình huấn luyện.

    Có thể nhận đầu vào là dict 'history' hoặc đường dẫn đến tệp JSON.
    """
    if isinstance(history_source, str) and os.path.exists(history_source):
        with open(history_source, "r") as f:
            hist = json.load(f)
    elif isinstance(history_source, dict):
        hist = history_source
    else:
        raise ValueError("Đầu vào phải là dict 'history' hoặc đường dẫn tệp JSON.")

    tr, vl = hist.get('train_loss', []), hist.get('val_loss', [])
    tr_ce, vl_ce = hist.get('train_ce', []), hist.get('val_ce', [])
    tr_dice, vl_dice = hist.get('train_dice', []), hist.get('val_dice', [])
    vi = hist.get('val_miou', [])
    epochs = range(1, len(tr) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Combined Loss
    axes[0, 0].plot(epochs, tr, 'b-', label='Train Loss', linewidth=2)
    axes[0, 0].plot(epochs, vl, 'r-', label='Val Loss', linewidth=2)
    axes[0, 0].set_title('Combined Loss (CE + Dice)')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend(); axes[0, 0].grid(True, ls='--', alpha=0.7)

    # Cross-Entropy Loss
    axes[0, 1].plot(epochs, tr_ce, 'b-', label='Train CE', linewidth=2)
    axes[0, 1].plot(epochs, vl_ce, 'r-', label='Val CE', linewidth=2)
    axes[0, 1].set_title('Cross-Entropy Loss')
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend(); axes[0, 1].grid(True, ls='--', alpha=0.7)

    # Dice Loss
    axes[1, 0].plot(epochs, tr_dice, 'b-', label='Train Dice', linewidth=2)
    axes[1, 0].plot(epochs, vl_dice, 'r-', label='Val Dice', linewidth=2)
    axes[1, 0].set_title('Dice Loss')
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend(); axes[1, 0].grid(True, ls='--', alpha=0.7)

    # Validation mIoU
    axes[1, 1].plot(epochs, vi, 'g-', label='Validation mIoU', linewidth=2)
    axes[1, 1].set_title('Validation mIoU')
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('mIoU')
    axes[1, 1].legend(); axes[1, 1].grid(True, ls='--', alpha=0.7)

    plt.tight_layout(); plt.show()


plot_training_history(history)


## 12. Trực quan hóa Dự đoán

Hiển thị đồng thời ảnh gốc, mặt nạ ground-truth, và dự đoán của mô hình với
bảng màu COCO. Việc so sánh trực quan giữa cột "Ground Truth" và "Dự đoán" cho
phép đánh giá định tính chất lượng phân vùng: mức độ khớp về biên đối tượng,
khả năng phân biệt giữa các lớp khác nhau, và các trường hợp nhầm lẫn điển hình.


In [ ]:
@torch.no_grad()
def visualize_predictions(model, dataloader, device, num_samples=8):
    """Hiển thị dự đoán của mô hình: ảnh gốc | GT | Dự đoán (màu)."""
    model.eval()
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225])
    fig, axes = plt.subplots(nrows=4, ncols=6, figsize=(20, 14))
    plt.subplots_adjust(wspace=0.1, hspace=0.3)
    shown = 0
    for images, masks in dataloader:
        images_gpu = images.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=device.type == 'cuda'):
            outputs = model(images_gpu)
        preds = outputs.argmax(dim=1).cpu()  # (B, H, W)
        images, masks = images.cpu(), masks.cpu()
        for i in range(images.size(0)):
            if shown >= num_samples:
                break
            row, col = shown // 2, (shown % 2) * 3
            img = np.clip(inv_normalize(images[i]).permute(1, 2, 0).numpy(), 0, 1)
            gt_mask = masks[i].numpy()
            pr_mask = preds[i].numpy()
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"Mẫu {shown+1}")
            axes[row, col].axis('off')
            axes[row, col+1].imshow(label_to_color(gt_mask))
            axes[row, col+1].set_title("Ground Truth")
            axes[row, col+1].axis('off')
            axes[row, col+2].imshow(label_to_color(pr_mask))
            axes[row, col+2].set_title("Dự đoán")
            axes[row, col+2].axis('off')
            shown += 1
        if shown >= num_samples:
            break
    plt.suptitle("U-MobileViT — Đánh giá Định tính (Ảnh | GT | Dự đoán)",
                 fontsize=16, y=0.92)
    plt.show()


## 13. Nạp Mô hình Tối ưu và Đánh giá Cuối cùng

Nạp checkpoint có `val_loss` thấp nhất (`best_u_mobilevit_net_10cls.pth`) và
trực quan hóa kết quả dự đoán trên cả tập huấn luyện và tập đánh giá. Bước này
cung cấp cái nhìn tổng quan cuối cùng về năng lực của mô hình sau quá trình
huấn luyện.


In [ ]:
best_path = os.path.join(SAVE_MODEL_DIR, "best_u_mobilevit_net_10cls.pth")
if os.path.exists(best_path):
    best_ckpt = torch.load(best_path, map_location=device, weights_only=True)
    model.load_state_dict(best_ckpt['model_state_dict'])
    print(f"Đã nạp mô hình tối ưu (epoch {best_ckpt.get('epoch')}, "
          f"val_mIoU={best_ckpt.get('val_miou'):.4f})")
else:
    print(f"[Cảnh báo] Chưa tìm thấy checkpoint tại {best_path}. "
          f"Sử dụng mô hình hiện tại trong bộ nhớ.")


In [ ]:
print("\nKết quả trên tập Huấn luyện:")
visualize_predictions(model, train_loader, device, num_samples=8)


In [ ]:
print("\nKết quả trên tập Đánh giá:")
visualize_predictions(model, val_loader, device, num_samples=8)
